# ShopSpy — E-Commerce Product Description Writer
## Phase 2: Fine-Tuning Llama 3.2 3B on Google Colab

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Have your `dataset.jsonl` file ready to upload
3. Have your Hugging Face token ready (huggingface.co/settings/tokens)
4. Request access to Llama 3.2: huggingface.co/meta-llama/Llama-3.2-3B-Instruct (usually instant)

## Step 1 — Install dependencies

In [ ]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets huggingface_hub
print('Done installing.')

## Step 2 — Log in to Hugging Face
Get your token at: huggingface.co/settings/tokens (create a token with **Read** access)

In [ ]:
from huggingface_hub import login

HF_TOKEN = ""  # ← PASTE YOUR TOKEN HERE
login(token=HF_TOKEN)
print('Logged in to Hugging Face.')

## Step 3 — Upload and load dataset

In [ ]:
from google.colab import files
import json

print('Upload your dataset.jsonl file...')
uploaded = files.upload()

data = []
with open('dataset.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            data.append(json.loads(line))

print(f'Loaded {len(data)} training examples.')
print('\nSample entry:')
print('INPUT:', data[0]['input'][:200])
print('OUTPUT:', data[0]['output'][:200])

## Step 4 — Format dataset for Llama 3.2

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = "You are a professional e-commerce copywriter. You write high-converting product descriptions that lead with benefits, use sensory language, and drive action."

def format_example(example):
    text = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n{example['instruction']}\n\n{example['input']}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n{example['output']}<|eot_id|>"
    )
    return {"text": text}

dataset = Dataset.from_list(data)
dataset = dataset.map(format_example)

# 90% train / 10% eval
split        = dataset.train_test_split(test_size=0.1, seed=42)
train_data   = split['train']
eval_data    = split['test']

print(f'Train: {len(train_data)} examples')
print(f'Eval:  {len(eval_data)} examples')

## Step 5 — Load Llama 3.2 3B in 4-bit quantization
This fits on a free T4 GPU (~15GB VRAM needed, T4 has 16GB).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN
)

print('Model loaded successfully!')
print(f'GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Step 6 — Configure LoRA
Only trains ~1% of parameters — that's what makes it fast and cheap.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expect: ~1-2% trainable — that's correct

## Step 7 — Train
Takes ~2 hours on a free T4 GPU. Watch the training loss — it should drop from ~2.5 toward ~0.8.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./shopspy-product-writer",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    dataset_text_field="text",
    max_seq_length=512,
    args=training_args
)

print('Starting training...')
trainer.train()
print('Training complete!')

## Step 8 — Test the model before pushing
Try it on a product it has never seen.

In [ ]:
from transformers import pipeline

test_prompt = (
    f"<|begin_of_text|>"
    f"<|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}<|eot_id|>"
    f"<|start_header_id|>user<|end_header_id|>\n"
    f"Write a high-converting product description for an e-commerce store.\n\n"
    f"Product: Pro Training Shorts - Black\nCategory: Mens Shorts\nBrand: ShopSpy Test Brand<|eot_id|>"
    f"<|start_header_id|>assistant<|end_header_id|>\n"
)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=300)
result = pipe(test_prompt)
output = result[0]['generated_text'].split('<|start_header_id|>assistant<|end_header_id|>')[-1]
print('--- Generated Description ---')
print(output.replace('<|eot_id|>', '').strip())

## Step 9 — Push to Hugging Face Hub
Replace `YOUR_HF_USERNAME` with your actual username.

In [ ]:
HF_USERNAME = ""  # ← YOUR HUGGING FACE USERNAME HERE
REPO_NAME   = "shopspy-product-writer"

model.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}", token=HF_TOKEN)
tokenizer.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}", token=HF_TOKEN)

print(f'Model live at: huggingface.co/{HF_USERNAME}/{REPO_NAME}')

## Done — Phase 2 Complete!

Your fine-tuned model is now on Hugging Face.

**Next steps (Phase 3):**
- Compare your model vs Claude on 50 products — score benefit clarity, emotional appeal, CTA strength
- Document which descriptions people prefer
- That comparison IS your portfolio piece

**Phase 4:**
- Add a 'Generate Description' button in ShopSpy dashboard
- Call your Hugging Face model endpoint
- Gate it behind the Premium plan tier